In [3]:
# ============================================================
# MAIZE DEPLOYMENT — CELL 1
# ENVIRONMENT, DRIVE AND DATASET
# ============================================================

!pip install -q --upgrade huggingface_hub

from google.colab import drive
from pathlib import Path

import shutil
import zipfile

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/final_year_project/maize")

DATASET_ROOT = Path("/content/Final_Maize_Dataset")
TRAIN_ROOT = DATASET_ROOT / "train"

CHECKPOINT_PATH = Path(
    "/content/drive/MyDrive/"
    "final_year_project/maize/checkpoints/"
    "maize_hybrid_stage2_best.pth"
)

DEPLOYMENT_DIR = Path(
    "/content/maize_deployment_v1"
)

DATASET_ZIP_NAME = "Final_Maize_Dataset.zip"

EXPECTED_COUNTS = {
    "Healthy": 1662,
    "MLN": 1671,
    "MSV": 1709,
    "NLB": 341,
    "NLS": 379
}

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

# ------------------------------------------------------------
# Extract dataset only when it is unavailable
# ------------------------------------------------------------

if not TRAIN_ROOT.exists():

    print("Dataset is not currently extracted.")

    matches = list(
        DRIVE_ROOT.rglob(DATASET_ZIP_NAME)
    )

    if not matches:
        raise FileNotFoundError(
            f"{DATASET_ZIP_NAME} was not found in Google Drive."
        )

    dataset_zip = matches[0]

    print("Using dataset ZIP:")
    print(dataset_zip)

    with zipfile.ZipFile(dataset_zip, "r") as zip_file:

        bad_file = zip_file.testzip()

        if bad_file is not None:
            raise RuntimeError(
                f"Corrupted ZIP member: {bad_file}"
            )

        zip_file.extractall("/content")

# ------------------------------------------------------------
# Verify paths
# ------------------------------------------------------------

assert TRAIN_ROOT.exists(), (
    f"Training folder not found: {TRAIN_ROOT}"
)

assert CHECKPOINT_PATH.exists(), (
    f"Stage-2 checkpoint not found: {CHECKPOINT_PATH}"
)

# ------------------------------------------------------------
# Verify exact training counts
# ------------------------------------------------------------

print("=" * 70)
print("TRAINING DATA VERIFICATION")
print("=" * 70)

total_images = 0

for class_name, expected_count in EXPECTED_COUNTS.items():

    class_dir = TRAIN_ROOT / class_name

    assert class_dir.exists(), (
        f"Missing class folder: {class_dir}"
    )

    files = [
        path
        for path in class_dir.iterdir()
        if (
            path.is_file()
            and path.suffix.lower() in IMAGE_EXTENSIONS
        )
    ]

    found_count = len(files)

    print(
        f"{class_name:<10}: "
        f"{found_count:,}"
    )

    assert found_count == expected_count, (
        f"{class_name}: expected {expected_count}, "
        f"found {found_count}"
    )

    total_images += found_count

assert total_images == 5762

# ------------------------------------------------------------
# Create a fresh package folder
# ------------------------------------------------------------

if DEPLOYMENT_DIR.exists():
    shutil.rmtree(DEPLOYMENT_DIR)

DEPLOYMENT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\n✓ Dataset verified")
print("✓ Stage-2 checkpoint found")
print("✓ Deployment directory created")
print(DEPLOYMENT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset is not currently extracted.
Using dataset ZIP:
/content/drive/MyDrive/final_year_project/maize/Final_Maize_Dataset.zip
TRAINING DATA VERIFICATION
Healthy   : 1,662
MLN       : 1,671
MSV       : 1,709
NLB       : 341
NLS       : 379

✓ Dataset verified
✓ Stage-2 checkpoint found
✓ Deployment directory created
/content/maize_deployment_v1


In [4]:
# ============================================================
# MAIZE DEPLOYMENT — CELL 2
# PRODUCTION MODEL DEFINITION
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision.models import resnet50

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

CLASS_NAMES = [
    "Healthy",
    "MLN",
    "MSV",
    "NLB",
    "NLS"
]

CLASS_TO_IDX = {
    "Healthy": 0,
    "MLN": 1,
    "MSV": 2,
    "NLB": 3,
    "NLS": 4
}

IMAGE_SIZE = 256
RESIZE_SIZE = 288

EMBED_DIM = 512
NUM_HEADS = 8
NUM_TRANSFORMER_LAYERS = 4
MLP_DIM = 2048
DROPOUT = 0.20
NUM_CLASSES = 5

CALIBRATION_TEMPERATURE = 0.740006

KNN_K = 1
KNN_THRESHOLD = 0.99242216


class ResNetTransformerHybrid(nn.Module):

    def __init__(
        self,
        num_classes=5,
        embed_dim=512,
        num_heads=8,
        num_layers=4,
        mlp_dim=2048,
        dropout=0.20
    ):

        super().__init__()

        # No pretrained download is required during deployment.
        backbone = resnet50(
            weights=None
        )

        self.backbone = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4
        )

        self.projection = nn.Conv2d(
            in_channels=2048,
            out_channels=embed_dim,
            kernel_size=1
        )

        # 256 / 32 = 8
        # 8 × 8 = 64 Transformer tokens.
        self.num_tokens = 64

        self.pos_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.num_tokens,
                embed_dim
            )
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=mlp_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.norm = nn.LayerNorm(
            embed_dim
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.30),
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(256, num_classes)
        )

    def forward_features(self, x):

        x = self.backbone(x)
        x = self.projection(x)

        x = x.flatten(2)
        x = x.transpose(1, 2)

        if x.shape[1] != self.num_tokens:
            raise ValueError(
                "Unexpected number of image tokens. "
                f"Expected {self.num_tokens}, "
                f"received {x.shape[1]}. "
                "Use Resize(288) followed by CenterCrop(256)."
            )

        x = x + self.pos_embedding
        x = self.transformer(x)
        x = self.norm(x)

        embedding = x.mean(dim=1)

        return embedding

    def forward(
        self,
        x,
        return_embedding=False
    ):

        embedding = self.forward_features(x)

        logits = self.classifier(
            embedding
        )

        if return_embedding:

            normalized_embedding = F.normalize(
                embedding,
                p=2,
                dim=1
            )

            return logits, normalized_embedding

        return logits


# ------------------------------------------------------------
# Load Stage-2 checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint stage:", checkpoint.get("stage"))
print("Checkpoint epoch:", checkpoint.get("epoch"))
print("Architecture:", checkpoint.get("architecture"))
print("Image size:", checkpoint.get("image_size"))
print("Class mapping:", checkpoint.get("class_to_idx"))

assert checkpoint["class_to_idx"] == CLASS_TO_IDX
assert checkpoint["image_size"] == IMAGE_SIZE

model = ResNetTransformerHybrid(
    num_classes=NUM_CLASSES,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_TRANSFORMER_LAYERS,
    mlp_dim=MLP_DIM,
    dropout=DROPOUT
)

load_result = model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

print("\nCheckpoint loading result:")
print(load_result)

model = model.to(DEVICE)
model.eval()

for parameter in model.parameters():
    parameter.requires_grad = False

print("\n✓ Stage-2 model loaded")
print("✓ All checkpoint keys matched")
print("✓ Model frozen")
print("✓ Device:", DEVICE)

Checkpoint stage: 2
Checkpoint epoch: 2
Architecture: ResNet50_Transformer_Hybrid
Image size: 256
Class mapping: {'Healthy': 0, 'MLN': 1, 'MSV': 2, 'NLB': 3, 'NLS': 4}


/tmp/ipykernel_1510/536491888.py:108: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(



Checkpoint loading result:
<All keys matched successfully>

✓ Stage-2 model loaded
✓ All checkpoint keys matched
✓ Model frozen
✓ Device: cuda


In [5]:
# ============================================================
# MAIZE DEPLOYMENT — CELL 3
# REBUILD TRAINING REFERENCE EMBEDDINGS
# ============================================================

import numpy as np

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225
]

inference_transform = transforms.Compose([
    transforms.Resize(
        RESIZE_SIZE
    ),

    transforms.CenterCrop(
        IMAGE_SIZE
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

reference_dataset = datasets.ImageFolder(
    TRAIN_ROOT,
    transform=inference_transform
)

assert reference_dataset.class_to_idx == CLASS_TO_IDX
assert len(reference_dataset) == 5762

reference_loader = DataLoader(
    reference_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=(DEVICE.type == "cuda")
)

embedding_batches = []
label_batches = []

print("=" * 70)
print("EXTRACTING PRODUCTION REFERENCE EMBEDDINGS")
print("=" * 70)

model.eval()

with torch.inference_mode():

    processed = 0

    for batch_number, (
        images,
        labels
    ) in enumerate(
        reference_loader,
        start=1
    ):

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        _, embeddings = model(
            images,
            return_embedding=True
        )

        embedding_batches.append(
            embeddings.cpu().numpy()
        )

        label_batches.append(
            labels.numpy()
        )

        processed += len(images)

        if (
            batch_number % 20 == 0
            or processed == len(reference_dataset)
        ):

            print(
                f"Processed "
                f"{processed:,}/"
                f"{len(reference_dataset):,}"
            )

reference_embeddings = np.concatenate(
    embedding_batches,
    axis=0
).astype(np.float32)

reference_labels = np.concatenate(
    label_batches,
    axis=0
).astype(np.int64)

# Normalize again as a safety check.
reference_norms = np.linalg.norm(
    reference_embeddings,
    axis=1,
    keepdims=True
)

reference_embeddings = (
    reference_embeddings
    /
    np.clip(
        reference_norms,
        1e-12,
        None
    )
).astype(np.float32)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

assert reference_embeddings.shape == (
    5762,
    512
)

assert reference_labels.shape == (
    5762,
)

assert np.isfinite(
    reference_embeddings
).all()

assert np.allclose(
    np.linalg.norm(
        reference_embeddings,
        axis=1
    ),
    1.0,
    atol=1e-5
)

print("\nReference-bank shape:")
print(reference_embeddings.shape)

print("\nReference labels:")

for class_index, class_name in enumerate(
    CLASS_NAMES
):

    count = int(
        np.sum(
            reference_labels == class_index
        )
    )

    print(
        f"{class_name:<10}: "
        f"{count:,}"
    )

    assert count == EXPECTED_COUNTS[class_name]

# ------------------------------------------------------------
# Save package assets
# ------------------------------------------------------------

np.save(
    DEPLOYMENT_DIR /
    "reference_embeddings.npy",
    reference_embeddings
)

np.save(
    DEPLOYMENT_DIR /
    "reference_labels.npy",
    reference_labels
)

# Save a clean state_dict rather than the full training
# checkpoint with optimizer state.
cpu_state_dict = {
    key: value.detach().cpu()
    for key, value
    in model.state_dict().items()
}

torch.save(
    cpu_state_dict,
    DEPLOYMENT_DIR /
    "model_weights.pth"
)

print("\n✓ Reference bank saved")
print("✓ Reference labels saved")
print("✓ Production model weights saved")

EXTRACTING PRODUCTION REFERENCE EMBEDDINGS
Processed 640/5,762
Processed 1,280/5,762
Processed 1,920/5,762
Processed 2,560/5,762
Processed 3,200/5,762
Processed 3,840/5,762
Processed 4,480/5,762
Processed 5,120/5,762
Processed 5,760/5,762
Processed 5,762/5,762

Reference-bank shape:
(5762, 512)

Reference labels:
Healthy   : 1,662
MLN       : 1,671
MSV       : 1,709
NLB       : 341
NLS       : 379

✓ Reference bank saved
✓ Reference labels saved
✓ Production model weights saved


In [6]:
# ============================================================
# MAIZE DEPLOYMENT — CELL 4
# CREATE model.py
# ============================================================

MODEL_PY = r'''
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision.models import resnet50


class ResNetTransformerHybrid(nn.Module):

    def __init__(
        self,
        num_classes=5,
        embed_dim=512,
        num_heads=8,
        num_layers=4,
        mlp_dim=2048,
        dropout=0.20
    ):

        super().__init__()

        backbone = resnet50(
            weights=None
        )

        self.backbone = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4
        )

        self.projection = nn.Conv2d(
            in_channels=2048,
            out_channels=embed_dim,
            kernel_size=1
        )

        self.num_tokens = 64

        self.pos_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.num_tokens,
                embed_dim
            )
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=mlp_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.norm = nn.LayerNorm(
            embed_dim
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.30),
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(256, num_classes)
        )

    def forward_features(self, x):

        x = self.backbone(x)
        x = self.projection(x)

        x = x.flatten(2)
        x = x.transpose(1, 2)

        if x.shape[1] != self.num_tokens:

            raise ValueError(
                "Incorrect input dimensions. "
                f"Expected {self.num_tokens} tokens, "
                f"received {x.shape[1]}."
            )

        x = x + self.pos_embedding
        x = self.transformer(x)
        x = self.norm(x)

        return x.mean(dim=1)

    def forward(
        self,
        x,
        return_embedding=False
    ):

        embedding = self.forward_features(x)

        logits = self.classifier(
            embedding
        )

        if return_embedding:

            normalized_embedding = F.normalize(
                embedding,
                p=2,
                dim=1
            )

            return logits, normalized_embedding

        return logits
'''

(
    DEPLOYMENT_DIR /
    "model.py"
).write_text(
    MODEL_PY.strip() + "\n",
    encoding="utf-8"
)

print("✓ model.py created")

✓ model.py created


In [7]:
# ============================================================
# MAIZE DEPLOYMENT — CELL 5
# CREATE inference.py
# ============================================================

INFERENCE_PY = r'''
import json

from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

from PIL import Image
from torchvision import transforms

try:
    from .model import ResNetTransformerHybrid
except ImportError:
    from model import ResNetTransformerHybrid


class MaizeDiseasePredictor:

    def __init__(
        self,
        model_dir=None,
        device=None
    ):

        self.model_dir = Path(
            model_dir
            if model_dir is not None
            else Path(__file__).resolve().parent
        )

        config_path = (
            self.model_dir /
            "config.json"
        )

        with config_path.open(
            "r",
            encoding="utf-8"
        ) as file:

            self.config = json.load(
                file
            )

        if device is None:

            device = (
                "cuda"
                if torch.cuda.is_available()
                else "cpu"
            )

        self.device = torch.device(
            device
        )

        self.class_names = self.config[
            "classes"
        ]

        self.temperature = float(
            self.config["temperature"]
        )

        self.knn_k = int(
            self.config["ood"]["k"]
        )

        self.knn_threshold = float(
            self.config["ood"]["threshold"]
        )

        self.model = ResNetTransformerHybrid(
            num_classes=len(self.class_names),
            embed_dim=self.config[
                "architecture_config"
            ]["embed_dim"],
            num_heads=self.config[
                "architecture_config"
            ]["num_heads"],
            num_layers=self.config[
                "architecture_config"
            ]["num_transformer_layers"],
            mlp_dim=self.config[
                "architecture_config"
            ]["mlp_dim"],
            dropout=self.config[
                "architecture_config"
            ]["dropout"]
        )

        weights_path = (
            self.model_dir /
            "model_weights.pth"
        )

        try:

            state_dict = torch.load(
                weights_path,
                map_location="cpu",
                weights_only=True
            )

        except TypeError:

            state_dict = torch.load(
                weights_path,
                map_location="cpu"
            )

        self.model.load_state_dict(
            state_dict,
            strict=True
        )

        self.model = self.model.to(
            self.device
        )

        self.model.eval()

        for parameter in self.model.parameters():
            parameter.requires_grad = False

        self.reference_embeddings = np.load(
            self.model_dir /
            "reference_embeddings.npy"
        ).astype(np.float32)

        self.reference_labels = np.load(
            self.model_dir /
            "reference_labels.npy"
        ).astype(np.int64)

        if self.reference_embeddings.ndim != 2:

            raise ValueError(
                "Reference embeddings must be a 2D array."
            )

        expected_dimension = int(
            self.config[
                "embedding_dimension"
            ]
        )

        if (
            self.reference_embeddings.shape[1]
            != expected_dimension
        ):

            raise ValueError(
                "Incorrect reference embedding dimension."
            )

        norms = np.linalg.norm(
            self.reference_embeddings,
            axis=1,
            keepdims=True
        )

        self.reference_embeddings = (
            self.reference_embeddings
            /
            np.clip(
                norms,
                1e-12,
                None
            )
        ).astype(np.float32)

        normalization = self.config[
            "normalization"
        ]

        self.transform = transforms.Compose([
            transforms.Resize(
                self.config["resize_size"]
            ),

            transforms.CenterCrop(
                self.config["input_size"]
            ),

            transforms.ToTensor(),

            transforms.Normalize(
                mean=normalization["mean"],
                std=normalization["std"]
            )
        ])

    def _load_image(self, image):

        if isinstance(image, Image.Image):

            return image.convert("RGB")

        if isinstance(
            image,
            (str, Path)
        ):

            with Image.open(image) as loaded:
                return loaded.convert("RGB")

        raise TypeError(
            "image must be a PIL image or file path"
        )

    def _calculate_knn_score(
        self,
        query_embedding
    ):

        query_embedding = np.asarray(
            query_embedding,
            dtype=np.float32
        ).reshape(1, -1)

        query_norm = np.linalg.norm(
            query_embedding,
            axis=1,
            keepdims=True
        )

        query_embedding = (
            query_embedding
            /
            np.clip(
                query_norm,
                1e-12,
                None
            )
        )

        similarities = (
            query_embedding
            @
            self.reference_embeddings.T
        )[0]

        top_k_scores = np.partition(
            similarities,
            -self.knn_k
        )[-self.knn_k:]

        return float(
            np.mean(top_k_scores)
        )

    @torch.inference_mode()
    def predict(self, image):

        pil_image = self._load_image(
            image
        )

        image_tensor = self.transform(
            pil_image
        ).unsqueeze(0).to(
            self.device
        )

        logits, embedding = self.model(
            image_tensor,
            return_embedding=True
        )

        calibrated_logits = (
            logits /
            self.temperature
        )

        probabilities = F.softmax(
            calibrated_logits,
            dim=1
        )

        top_count = min(
            3,
            len(self.class_names)
        )

        top_probabilities, top_indices = torch.topk(
            probabilities,
            k=top_count,
            dim=1
        )

        predicted_index = int(
            top_indices[0, 0].item()
        )

        confidence = float(
            top_probabilities[0, 0].item()
        )

        second_confidence = float(
            top_probabilities[0, 1].item()
        )

        probability_margin = float(
            confidence -
            second_confidence
        )

        reliability_score = (
            self._calculate_knn_score(
                embedding[0].cpu().numpy()
            )
        )

        accepted = (
            reliability_score
            >=
            self.knn_threshold
        )

        base_result = {
            "crop":
                self.config["crop"],

            "model_version":
                self.config["version"],

            "decision":
                "ACCEPT"
                if accepted
                else "UNCERTAIN",

            "predicted_class":
                self.class_names[
                    predicted_index
                ]
                if accepted
                else None,

            "confidence":
                round(confidence, 6),

            "probability_margin":
                round(
                    probability_margin,
                    6
                ),

            "reliability_score":
                round(
                    reliability_score,
                    6
                ),

            "reliability_threshold":
                self.knn_threshold,

            "reason_codes":
                []
                if accepted
                else [
                    "UNSUPPORTED_OR_UNCERTAIN_CONDITION"
                ]
        }

        if accepted:

            base_result["top_predictions"] = [
                {
                    "class": self.class_names[
                        int(class_index)
                    ],
                    "probability": round(
                        float(probability),
                        6
                    )
                }
                for probability, class_index
                in zip(
                    top_probabilities[0].cpu().tolist(),
                    top_indices[0].cpu().tolist()
                )
            ]

        else:

            base_result["message"] = (
                "The image does not reliably match "
                "the five supported maize conditions."
            )

        return base_result
'''

(
    DEPLOYMENT_DIR /
    "inference.py"
).write_text(
    INFERENCE_PY.strip() + "\n",
    encoding="utf-8"
)

(
    DEPLOYMENT_DIR /
    "__init__.py"
).write_text(
    (
        "from .inference import "
        "MaizeDiseasePredictor\n"
    ),
    encoding="utf-8"
)

print("✓ inference.py created")
print("✓ __init__.py created")

✓ inference.py created
✓ __init__.py created


In [9]:
# ============================================================
# MAIZE DEPLOYMENT — CELL 6
# CONFIGURATION AND DOCUMENTATION
# ============================================================

import json


# ============================================================
# 1. USER-FACING CLASS NAMES
# ============================================================

CLASS_DISPLAY_NAMES = {
    "Healthy": "Healthy",
    "MLN": "Maize Lethal Necrosis",
    "MSV": "Maize Streak Virus Disease",
    "NLB": "Northern Leaf Blight",
    "NLS": "Northern Leaf Spot"
}


# ============================================================
# 2. MODEL CONFIGURATION
# ============================================================

configuration = {
    "crop": "maize",

    "version": "1.0.0",

    "architecture": "ResNet50TransformerHybrid",

    "architecture_config": {
        "embed_dim": 512,
        "num_heads": 8,
        "num_transformer_layers": 4,
        "mlp_dim": 2048,
        "dropout": 0.20
    },

    "input_size": 256,

    "resize_size": 288,

    "embedding_dimension": 512,

    # Internal labels used by the trained model.
    "classes": CLASS_NAMES,

    # Original ImageFolder mapping.
    "class_to_idx": CLASS_TO_IDX,

    # Names shown to mobile users.
    "class_display_names": CLASS_DISPLAY_NAMES,

    "temperature": CALIBRATION_TEMPERATURE,

    "normalization": {
        "mean": IMAGENET_MEAN,
        "std": IMAGENET_STD
    },

    "ood": {
        "method": "cosine_knn",
        "k": KNN_K,
        "threshold": KNN_THRESHOLD,
        "reference_split": "training",
        "reference_count": 5762,
        "known_validation_retention": 0.9895
    },

    "supported_conditions": [
        {
            "code": "Healthy",
            "display_name": "Healthy"
        },
        {
            "code": "MLN",
            "display_name": "Maize Lethal Necrosis"
        },
        {
            "code": "MSV",
            "display_name": "Maize Streak Virus Disease"
        },
        {
            "code": "NLB",
            "display_name": "Northern Leaf Blight"
        },
        {
            "code": "NLS",
            "display_name": "Northern Leaf Spot"
        }
    ],

    "unsupported_behavior": (
        "Return UNCERTAIN and suppress the disease "
        "class code and display name."
    )
}


# ============================================================
# 3. SAVE config.json
# ============================================================

config_path = (
    DEPLOYMENT_DIR /
    "config.json"
)

with config_path.open(
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        configuration,
        file,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 4. CREATE requirements.txt
# ============================================================

requirements = (
    "torch\n"
    "torchvision\n"
    "numpy\n"
    "Pillow\n"
)

requirements_path = (
    DEPLOYMENT_DIR /
    "requirements.txt"
)

requirements_path.write_text(
    requirements,
    encoding="utf-8"
)


# ============================================================
# 5. CREATE README.md
#
# This version intentionally avoids nested Markdown code fences.
# ============================================================

readme_lines = [
    "---",
    "library_name: pytorch",
    "pipeline_tag: image-classification",
    "tags:",
    "- agriculture",
    "- plant-disease",
    "- maize",
    "- open-set-recognition",
    "- pytorch",
    "---",
    "",
    "# Maize Leaf Disease Model V1",
    "",
    "A ResNet50 plus Transformer/self-attention model for",
    "five supported maize leaf conditions.",
    "",
    "## Supported conditions",
    "",
    "| Internal code | Display name |",
    "|---|---|",
    "| Healthy | Healthy |",
    "| MLN | Maize Lethal Necrosis |",
    "| MSV | Maize Streak Virus Disease |",
    "| NLB | Northern Leaf Blight |",
    "| NLS | Northern Leaf Spot |",
    "",
    "Internal codes match the labels used during model training.",
    "Applications should show the full display name to users.",
    "",
    "## Input preprocessing",
    "",
    "- RGB image",
    "- Resize the shorter edge to 288 pixels",
    "- Center crop to 256 x 256 pixels",
    "- Convert the image to a PyTorch tensor",
    "- Apply ImageNet normalization",
    "",
    "Normalization mean: [0.485, 0.456, 0.406]",
    "",
    "Normalization standard deviation:",
    "[0.229, 0.224, 0.225]",
    "",
    "## Architecture",
    "",
    "- ResNet50 convolutional backbone",
    "- 2048-to-512 feature projection",
    "- Four Transformer encoder layers",
    "- Eight attention heads",
    "- Mean spatial-token pooling",
    "- Five-class classification head",
    "",
    "## Confidence calibration",
    "",
    "Temperature scaling is applied before Softmax.",
    "",
    "Temperature: 0.740006",
    "",
    "## Unsupported-condition rejection",
    "",
    "The package uses cosine nearest-neighbour similarity",
    "against 5,762 frozen training embeddings.",
    "",
    "Method: Cosine kNN",
    "",
    "K: 1",
    "",
    "Threshold: 0.99242216",
    "",
    "When similarity is below the threshold, the package",
    "returns UNCERTAIN and suppresses the predicted class.",
    "",
    "## Accepted response fields",
    "",
    "- crop",
    "- model_version",
    "- decision",
    "- predicted_class_code",
    "- predicted_class",
    "- confidence",
    "- probability_margin",
    "- reliability_score",
    "- top_predictions",
    "",
    "Example accepted class:",
    "",
    "predicted_class_code: NLB",
    "",
    "predicted_class: Northern Leaf Blight",
    "",
    "## Important limitation",
    "",
    "This package performs supported-disease classification",
    "and embedding-similarity rejection.",
    "",
    "The complete application should run separate plant/non-plant",
    "and crop-identity validators before calling this model.",
    ""
]

readme = "\n".join(
    readme_lines
)

readme_path = (
    DEPLOYMENT_DIR /
    "README.md"
)

readme_path.write_text(
    readme,
    encoding="utf-8"
)


# ============================================================
# 6. VERIFY SAVED CONFIGURATION
# ============================================================

with config_path.open(
    "r",
    encoding="utf-8"
) as file:

    saved_configuration = json.load(
        file
    )


assert saved_configuration[
    "classes"
] == CLASS_NAMES

assert saved_configuration[
    "class_to_idx"
] == CLASS_TO_IDX

assert saved_configuration[
    "class_display_names"
] == CLASS_DISPLAY_NAMES

assert saved_configuration[
    "class_display_names"
]["Healthy"] == "Healthy"

assert saved_configuration[
    "class_display_names"
]["MLN"] == "Maize Lethal Necrosis"

assert saved_configuration[
    "class_display_names"
]["MSV"] == "Maize Streak Virus Disease"

assert saved_configuration[
    "class_display_names"
]["NLB"] == "Northern Leaf Blight"

assert saved_configuration[
    "class_display_names"
]["NLS"] == "Northern Leaf Spot"


# ============================================================
# 7. VERIFY CREATED FILES
# ============================================================

assert config_path.exists()
assert requirements_path.exists()
assert readme_path.exists()

assert config_path.stat().st_size > 0
assert requirements_path.stat().st_size > 0
assert readme_path.stat().st_size > 0


# ============================================================
# 8. FINAL OUTPUT
# ============================================================

print("=" * 75)
print("CONFIGURATION AND DOCUMENTATION CREATED")
print("=" * 75)

print(f"\n✓ {config_path}")
print(f"✓ {requirements_path}")
print(f"✓ {readme_path}")

print("\nClass display-name mapping:")

for class_code in CLASS_NAMES:

    display_name = (
        CLASS_DISPLAY_NAMES[
            class_code
        ]
    )

    print(
        f"  {class_code:<8} -> "
        f"{display_name}"
    )

print(
    "\n✓ Internal model labels remain unchanged"
)

print(
    "✓ Full disease names added for mobile display"
)

print(
    "✓ Configuration verification passed"
)

CONFIGURATION AND DOCUMENTATION CREATED

✓ /content/maize_deployment_v1/config.json
✓ /content/maize_deployment_v1/requirements.txt
✓ /content/maize_deployment_v1/README.md

Class display-name mapping:
  Healthy  -> Healthy
  MLN      -> Maize Lethal Necrosis
  MSV      -> Maize Streak Virus Disease
  NLB      -> Northern Leaf Blight
  NLS      -> Northern Leaf Spot

✓ Internal model labels remain unchanged
✓ Full disease names added for mobile display
✓ Configuration verification passed


In [10]:
# ============================================================
# MAIZE DEPLOYMENT — UPDATED CELL 5
# CREATE inference.py WITH FULL DISEASE NAMES
# ============================================================

INFERENCE_PY = r'''
import json

from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

from PIL import Image
from torchvision import transforms

try:
    from .model import ResNetTransformerHybrid
except ImportError:
    from model import ResNetTransformerHybrid


class MaizeDiseasePredictor:

    def __init__(
        self,
        model_dir=None,
        device=None
    ):

        # ====================================================
        # 1. MODEL DIRECTORY
        # ====================================================

        self.model_dir = Path(
            model_dir
            if model_dir is not None
            else Path(__file__).resolve().parent
        )

        if not self.model_dir.exists():

            raise FileNotFoundError(
                f"Model directory not found: "
                f"{self.model_dir}"
            )

        # ====================================================
        # 2. LOAD CONFIGURATION
        # ====================================================

        config_path = (
            self.model_dir /
            "config.json"
        )

        if not config_path.exists():

            raise FileNotFoundError(
                f"Configuration file not found: "
                f"{config_path}"
            )

        with config_path.open(
            "r",
            encoding="utf-8"
        ) as file:

            self.config = json.load(
                file
            )

        # Internal model labels:
        # Healthy, MLN, MSV, NLB, NLS
        self.class_names = self.config[
            "classes"
        ]

        # User-facing full disease names.
        self.class_display_names = (
            self.config[
                "class_display_names"
            ]
        )

        # Verify that every internal class has a display name.
        for class_code in self.class_names:

            if (
                class_code
                not in self.class_display_names
            ):

                raise ValueError(
                    "Missing display name for "
                    f"class: {class_code}"
                )

        self.temperature = float(
            self.config[
                "temperature"
            ]
        )

        self.knn_k = int(
            self.config[
                "ood"
            ][
                "k"
            ]
        )

        self.knn_threshold = float(
            self.config[
                "ood"
            ][
                "threshold"
            ]
        )

        # ====================================================
        # 3. SELECT DEVICE
        # ====================================================

        if device is None:

            device = (
                "cuda"
                if torch.cuda.is_available()
                else "cpu"
            )

        self.device = torch.device(
            device
        )

        # ====================================================
        # 4. CREATE MODEL
        # ====================================================

        architecture_config = (
            self.config[
                "architecture_config"
            ]
        )

        self.model = ResNetTransformerHybrid(
            num_classes=len(
                self.class_names
            ),

            embed_dim=int(
                architecture_config[
                    "embed_dim"
                ]
            ),

            num_heads=int(
                architecture_config[
                    "num_heads"
                ]
            ),

            num_layers=int(
                architecture_config[
                    "num_transformer_layers"
                ]
            ),

            mlp_dim=int(
                architecture_config[
                    "mlp_dim"
                ]
            ),

            dropout=float(
                architecture_config[
                    "dropout"
                ]
            )
        )

        # ====================================================
        # 5. LOAD MODEL WEIGHTS
        # ====================================================

        weights_path = (
            self.model_dir /
            "model_weights.pth"
        )

        if not weights_path.exists():

            raise FileNotFoundError(
                f"Model weights not found: "
                f"{weights_path}"
            )

        try:

            state_dict = torch.load(
                weights_path,
                map_location="cpu",
                weights_only=True
            )

        except TypeError:

            # Compatibility with older PyTorch versions.
            state_dict = torch.load(
                weights_path,
                map_location="cpu"
            )

        load_result = (
            self.model.load_state_dict(
                state_dict,
                strict=True
            )
        )

        self.model = self.model.to(
            self.device
        )

        self.model.eval()

        for parameter in self.model.parameters():
            parameter.requires_grad = False

        # ====================================================
        # 6. LOAD REFERENCE EMBEDDINGS
        # ====================================================

        reference_embeddings_path = (
            self.model_dir /
            "reference_embeddings.npy"
        )

        reference_labels_path = (
            self.model_dir /
            "reference_labels.npy"
        )

        if not reference_embeddings_path.exists():

            raise FileNotFoundError(
                "Reference embeddings not found: "
                f"{reference_embeddings_path}"
            )

        if not reference_labels_path.exists():

            raise FileNotFoundError(
                "Reference labels not found: "
                f"{reference_labels_path}"
            )

        self.reference_embeddings = np.load(
            reference_embeddings_path
        ).astype(
            np.float32
        )

        self.reference_labels = np.load(
            reference_labels_path
        ).astype(
            np.int64
        )

        # ====================================================
        # 7. VERIFY REFERENCE BANK
        # ====================================================

        if self.reference_embeddings.ndim != 2:

            raise ValueError(
                "Reference embeddings must be "
                "a two-dimensional array."
            )

        expected_dimension = int(
            self.config[
                "embedding_dimension"
            ]
        )

        if (
            self.reference_embeddings.shape[1]
            != expected_dimension
        ):

            raise ValueError(
                "Incorrect reference embedding "
                "dimension. "
                f"Expected {expected_dimension}, "
                f"received "
                f"{self.reference_embeddings.shape[1]}."
            )

        if (
            len(self.reference_embeddings)
            != len(self.reference_labels)
        ):

            raise ValueError(
                "Reference embeddings and labels "
                "have different lengths."
            )

        if (
            len(self.reference_embeddings)
            != int(
                self.config[
                    "ood"
                ][
                    "reference_count"
                ]
            )
        ):

            raise ValueError(
                "Reference-bank count does not "
                "match config.json."
            )

        if not np.isfinite(
            self.reference_embeddings
        ).all():

            raise ValueError(
                "Reference embeddings contain "
                "NaN or infinite values."
            )

        # Normalize the reference bank as a safety check.
        reference_norms = np.linalg.norm(
            self.reference_embeddings,
            axis=1,
            keepdims=True
        )

        self.reference_embeddings = (
            self.reference_embeddings
            /
            np.clip(
                reference_norms,
                1e-12,
                None
            )
        ).astype(
            np.float32
        )

        # ====================================================
        # 8. CREATE INFERENCE TRANSFORM
        # ====================================================

        normalization = self.config[
            "normalization"
        ]

        self.transform = transforms.Compose([
            transforms.Resize(
                int(
                    self.config[
                        "resize_size"
                    ]
                )
            ),

            transforms.CenterCrop(
                int(
                    self.config[
                        "input_size"
                    ]
                )
            ),

            transforms.ToTensor(),

            transforms.Normalize(
                mean=normalization[
                    "mean"
                ],

                std=normalization[
                    "std"
                ]
            )
        ])

    # ========================================================
    # LOAD INPUT IMAGE
    # ========================================================

    def _load_image(
        self,
        image
    ):

        if isinstance(
            image,
            Image.Image
        ):

            return image.convert(
                "RGB"
            )

        if isinstance(
            image,
            (str, Path)
        ):

            image_path = Path(
                image
            )

            if not image_path.exists():

                raise FileNotFoundError(
                    f"Image not found: "
                    f"{image_path}"
                )

            with Image.open(
                image_path
            ) as loaded_image:

                return loaded_image.convert(
                    "RGB"
                )

        raise TypeError(
            "image must be a PIL Image "
            "or a valid image-file path."
        )

    # ========================================================
    # CALCULATE COSINE kNN SCORE
    # ========================================================

    def _calculate_knn_score(
        self,
        query_embedding
    ):

        query_embedding = np.asarray(
            query_embedding,
            dtype=np.float32
        ).reshape(
            1,
            -1
        )

        query_norm = np.linalg.norm(
            query_embedding,
            axis=1,
            keepdims=True
        )

        query_embedding = (
            query_embedding
            /
            np.clip(
                query_norm,
                1e-12,
                None
            )
        )

        similarities = (
            query_embedding
            @
            self.reference_embeddings.T
        )[0]

        top_k_scores = np.partition(
            similarities,
            -self.knn_k
        )[
            -self.knn_k:
        ]

        knn_score = float(
            np.mean(
                top_k_scores
            )
        )

        return knn_score

    # ========================================================
    # PREDICT
    # ========================================================

    @torch.inference_mode()
    def predict(
        self,
        image
    ):

        # ----------------------------------------------------
        # Load and preprocess image
        # ----------------------------------------------------

        pil_image = self._load_image(
            image
        )

        image_tensor = self.transform(
            pil_image
        ).unsqueeze(
            0
        ).to(
            self.device
        )

        # ----------------------------------------------------
        # Classification and embedding extraction
        # ----------------------------------------------------

        logits, embedding = self.model(
            image_tensor,
            return_embedding=True
        )

        # ----------------------------------------------------
        # Temperature-scaled probabilities
        # ----------------------------------------------------

        calibrated_logits = (
            logits /
            self.temperature
        )

        probabilities = F.softmax(
            calibrated_logits,
            dim=1
        )

        # ----------------------------------------------------
        # Top predictions
        # ----------------------------------------------------

        top_count = min(
            3,
            len(
                self.class_names
            )
        )

        (
            top_probabilities,
            top_indices
        ) = torch.topk(
            probabilities,
            k=top_count,
            dim=1
        )

        predicted_index = int(
            top_indices[
                0,
                0
            ].item()
        )

        predicted_class_code = (
            self.class_names[
                predicted_index
            ]
        )

        predicted_display_name = (
            self.class_display_names[
                predicted_class_code
            ]
        )

        confidence = float(
            top_probabilities[
                0,
                0
            ].item()
        )

        second_confidence = float(
            top_probabilities[
                0,
                1
            ].item()
        )

        probability_margin = float(
            confidence -
            second_confidence
        )

        # ----------------------------------------------------
        # Cosine kNN reliability
        # ----------------------------------------------------

        reliability_score = (
            self._calculate_knn_score(
                embedding[
                    0
                ].cpu().numpy()
            )
        )

        accepted = (
            reliability_score
            >=
            self.knn_threshold
        )

        # ----------------------------------------------------
        # Common result fields
        # ----------------------------------------------------

        result = {
            "crop":
                self.config[
                    "crop"
                ],

            "model_version":
                self.config[
                    "version"
                ],

            "decision":
                (
                    "ACCEPT"
                    if accepted
                    else "UNCERTAIN"
                ),

            # Suppress class information when uncertain.
            "predicted_class_code":
                (
                    predicted_class_code
                    if accepted
                    else None
                ),

            "predicted_class":
                (
                    predicted_display_name
                    if accepted
                    else None
                ),

            "confidence":
                round(
                    confidence,
                    6
                ),

            "probability_margin":
                round(
                    probability_margin,
                    6
                ),

            "reliability_score":
                round(
                    reliability_score,
                    6
                ),

            "reliability_threshold":
                self.knn_threshold,

            "reason_codes":
                (
                    []
                    if accepted
                    else [
                        "UNSUPPORTED_OR_UNCERTAIN_CONDITION"
                    ]
                )
        }

        # ----------------------------------------------------
        # Add top predictions only when accepted
        # ----------------------------------------------------

        if accepted:

            top_predictions = []

            for (
                probability,
                class_index
            ) in zip(
                top_probabilities[
                    0
                ].cpu().tolist(),

                top_indices[
                    0
                ].cpu().tolist()
            ):

                class_index = int(
                    class_index
                )

                class_code = (
                    self.class_names[
                        class_index
                    ]
                )

                display_name = (
                    self.class_display_names[
                        class_code
                    ]
                )

                top_predictions.append({
                    "class_code":
                        class_code,

                    "class_name":
                        display_name,

                    "probability":
                        round(
                            float(
                                probability
                            ),
                            6
                        )
                })

            result[
                "top_predictions"
            ] = top_predictions

            result[
                "message"
            ] = (
                f"Prediction accepted: "
                f"{predicted_display_name}"
            )

        else:

            result[
                "message"
            ] = (
                "The image does not reliably "
                "match the five supported "
                "maize conditions."
            )

        return result
'''

# ============================================================
# SAVE inference.py
# ============================================================

inference_path = (
    DEPLOYMENT_DIR /
    "inference.py"
)

inference_path.write_text(
    INFERENCE_PY.strip() + "\n",
    encoding="utf-8"
)


# ============================================================
# CREATE/UPDATE __init__.py
# ============================================================

init_path = (
    DEPLOYMENT_DIR /
    "__init__.py"
)

init_path.write_text(
    (
        "from .inference import "
        "MaizeDiseasePredictor\n"
    ),
    encoding="utf-8"
)


# ============================================================
# VERIFY CREATED FILE
# ============================================================

assert inference_path.exists()
assert inference_path.stat().st_size > 0

saved_inference_code = (
    inference_path.read_text(
        encoding="utf-8"
    )
)

assert (
    "class_display_names"
    in saved_inference_code
)

assert (
    "predicted_class_code"
    in saved_inference_code
)

assert (
    "predicted_class"
    in saved_inference_code
)

assert (
    "top_predictions"
    in saved_inference_code
)


# ============================================================
# FINAL OUTPUT
# ============================================================

print("=" * 75)
print("UPDATED INFERENCE FILE CREATED")
print("=" * 75)

print(f"\n✓ {inference_path}")
print(f"✓ {init_path}")

print(
    "\n✓ Full disease display names supported"
)

print(
    "✓ Internal class codes preserved"
)

print(
    "✓ Class output suppressed when uncertain"
)

print(
    "✓ Updated Cell 5 completed successfully"
)

UPDATED INFERENCE FILE CREATED

✓ /content/maize_deployment_v1/inference.py
✓ /content/maize_deployment_v1/__init__.py

✓ Full disease display names supported
✓ Internal class codes preserved
✓ Class output suppressed when uncertain
✓ Updated Cell 5 completed successfully


In [12]:

## Cell 7 — Verify the complete package


# ============================================================
# MAIZE DEPLOYMENT — CELL 7
# COMPLETE PACKAGE VERIFICATION
# ============================================================

import json
import sys
import importlib

required_files = [
    "__init__.py",
    "model.py",
    "inference.py",
    "model_weights.pth",
    "config.json",
    "reference_embeddings.npy",
    "reference_labels.npy",
    "requirements.txt",
    "README.md"
]

print("=" * 75)
print("DEPLOYMENT PACKAGE")
print("=" * 75)

for file_name in required_files:

    file_path = (
        DEPLOYMENT_DIR /
        file_name
    )

    assert file_path.exists(), (
        f"Missing deployment file: {file_name}"
    )

    print(
        f"{file_name:<30} "
        f"{file_path.stat().st_size / 1024 / 1024:>9.2f} MB"
    )

# ------------------------------------------------------------
# Import the packaged inference implementation
# ------------------------------------------------------------

sys.path.insert(
    0,
    str(DEPLOYMENT_DIR)
)

if "inference" in sys.modules:
    del sys.modules["inference"]

if "model" in sys.modules:
    del sys.modules["model"]

deployment_inference = importlib.import_module(
    "inference"
)

MaizeDiseasePredictor = (
    deployment_inference
    .MaizeDiseasePredictor
)

predictor = MaizeDiseasePredictor(
    model_dir=DEPLOYMENT_DIR,
    device=str(DEVICE)
)

# ------------------------------------------------------------
# Test using one known training image
# ------------------------------------------------------------

sample_image_path = sorted(
    [
        path
        for path in (
            TRAIN_ROOT /
            "Healthy"
        ).iterdir()
        if (
            path.is_file()
            and
            path.suffix.lower()
            in IMAGE_EXTENSIONS
        )
    ]
)[0]

result = predictor.predict(
    sample_image_path
)

print("\n" + "=" * 75)
print("TEST PREDICTION")
print("=" * 75)

print("Test image:")
print(sample_image_path)

print("\nResult:")

print(
    json.dumps(
        result,
        indent=2
    )
)

assert result["decision"] in {
    "ACCEPT",
    "UNCERTAIN"
}

assert (
    result["predicted_class"] is None
    or
    result["predicted_class"] in CLASS_NAMES
)

print("\n✓ Package imported successfully")
print("✓ Checkpoint loaded successfully")
print("✓ Reference bank loaded successfully")
print("✓ End-to-end prediction completed")

DEPLOYMENT PACKAGE
__init__.py                         0.00 MB
model.py                            0.00 MB
inference.py                        0.02 MB
model_weights.pth                 142.73 MB
config.json                         0.00 MB
reference_embeddings.npy           11.25 MB
reference_labels.npy                0.04 MB
requirements.txt                    0.00 MB
README.md                           0.00 MB


/content/maize_deployment_v1/model.py:63: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(



TEST PREDICTION
Test image:
/content/Final_Maize_Dataset/train/Healthy/1711617163472.jpg

Result:
{
  "crop": "maize",
  "model_version": "1.0.0",
  "decision": "ACCEPT",
  "predicted_class_code": "Healthy",
  "predicted_class": "Healthy",
  "confidence": 0.986815,
  "probability_margin": 0.983094,
  "reliability_score": 1.0,
  "reliability_threshold": 0.99242216,
  "reason_codes": [],
  "top_predictions": [
    {
      "class_code": "Healthy",
      "class_name": "Healthy",
      "probability": 0.986815
    },
    {
      "class_code": "MLN",
      "class_name": "Maize Lethal Necrosis",
      "probability": 0.003721
    },
    {
      "class_code": "NLB",
      "class_name": "Northern Leaf Blight",
      "probability": 0.003329
    }
  ],
  "message": "Prediction accepted: Healthy"
}

✓ Package imported successfully
✓ Checkpoint loaded successfully
✓ Reference bank loaded successfully
✓ End-to-end prediction completed


In [13]:
# ============================================================
# MAIZE DEPLOYMENT — CELL 8
# SAVE PACKAGE BACKUP TO GOOGLE DRIVE
# ============================================================

import shutil

BACKUP_ROOT = Path(
    "/content/drive/MyDrive/"
    "final_year_project/maize/deployment"
)

BACKUP_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

ZIP_BASE_PATH = (
    BACKUP_ROOT /
    "maize_model_v1"
)

zip_path = shutil.make_archive(
    base_name=str(ZIP_BASE_PATH),
    format="zip",
    root_dir=str(DEPLOYMENT_DIR)
)

print("✓ Deployment ZIP created:")
print(zip_path)

✓ Deployment ZIP created:
/content/drive/MyDrive/final_year_project/maize/deployment/maize_model_v1.zip


In [15]:
# ============================================================
# MAIZE DEPLOYMENT — CELL 9
# UPLOAD maize/v1 TO HUGGING FACE
# ============================================================

from google.colab import userdata

from huggingface_hub import (
    HfApi,
    login
)

HF_REPO_ID = (
    "viraj1920/plant-disease-models"
)

HF_PATH_IN_REPO = "maize/v1"

HF_TOKEN = userdata.get(
    "HF_maize_token"
)

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found in Colab Secrets."
    )

login(
    token=HF_TOKEN,
    add_to_git_credential=False
)

api = HfApi(
    token=HF_TOKEN
)

# Safe if the repository already exists.
api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type="model",
    exist_ok=True,
    private=False
)

upload_result = api.upload_folder(
    folder_path=str(
        DEPLOYMENT_DIR
    ),

    repo_id=HF_REPO_ID,

    repo_type="model",

    path_in_repo=HF_PATH_IN_REPO,

    commit_message=(
        "Add Maize disease model v1 "
        "with calibrated kNN rejection"
    )
)

print("=" * 75)
print("UPLOAD COMPLETE")
print("=" * 75)

print(upload_result)

print(
    "\nModel folder:\n"
    f"https://huggingface.co/"
    f"{HF_REPO_ID}/tree/main/"
    f"{HF_PATH_IN_REPO}"
)

UPLOAD COMPLETE
https://huggingface.co/viraj1920/plant-disease-models/commit/ce07f82ffb79c973e75b1de26bea22691059b88a

Model folder:
https://huggingface.co/viraj1920/plant-disease-models/tree/main/maize/v1


In [19]:
# ============================================================
# MAIZE V1 — COMPLETE REMOTE HUGGING FACE SMOKE TEST
# RUN IN THE SAME DEPLOYMENT NOTEBOOK
# ============================================================

!pip install -q --upgrade huggingface_hub

import sys
import json
import shutil
import importlib

from pathlib import Path

from google.colab import (
    userdata,
    files
)

from huggingface_hub import (
    HfApi,
    login,
    snapshot_download
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

HF_REPO_ID = (
    "viraj1920/plant-disease-models"
)

REMOTE_TEST_ROOT = Path(
    "/content/maize_hf_remote_test"
)

REMOTE_MAIZE_DIR = (
    REMOTE_TEST_ROOT /
    "maize" /
    "v1"
)


# ============================================================
# 2. LOAD HUGGING FACE TOKEN
# ============================================================

try:

    HF_TOKEN = userdata.get(
        "HF_maize_token"
    )

except Exception as error:

    print(
        "HF_TOKEN could not be accessed."
    )

    print(
        "Open Colab Secrets, add HF_TOKEN, "
        "and enable notebook access."
    )

    raise error


if not HF_TOKEN:

    raise ValueError(
        "HF_TOKEN exists but is empty."
    )


print("✓ HF_TOKEN found")
print("✓ Token value remains hidden")


# ============================================================
# 3. AUTHENTICATE
# ============================================================

login(
    token=HF_TOKEN,
    add_to_git_credential=False
)

api = HfApi(
    token=HF_TOKEN
)

account_information = api.whoami()

authenticated_username = (
    account_information.get(
        "name",
        "Unknown"
    )
)

print(
    "\nAuthenticated account:",
    authenticated_username
)


# ============================================================
# 4. VERIFY REPOSITORY ACCESS
# ============================================================

repository_information = api.model_info(
    repo_id=HF_REPO_ID,
    token=HF_TOKEN
)

print(
    "Repository:",
    repository_information.id
)


# ============================================================
# 5. REMOVE ONLY THE OLD REMOTE TEST DIRECTORY
#
# This does not remove the deployment package or dataset.
# ============================================================

if REMOTE_TEST_ROOT.exists():

    shutil.rmtree(
        REMOTE_TEST_ROOT
    )

REMOTE_TEST_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 6. DOWNLOAD A FRESH REMOTE COPY
# ============================================================

print("\nDownloading maize/v1 from Hugging Face...")

snapshot_path = snapshot_download(
    repo_id=HF_REPO_ID,

    repo_type="model",

    revision="main",

    allow_patterns=[
        "maize/v1/*"
    ],

    local_dir=REMOTE_TEST_ROOT,

    token=HF_TOKEN,

    force_download=True
)


# ============================================================
# 7. VERIFY DOWNLOADED FILES
# ============================================================

assert REMOTE_MAIZE_DIR.exists(), (
    f"Remote Maize directory not found: "
    f"{REMOTE_MAIZE_DIR}"
)

required_files = [
    "__init__.py",
    "model.py",
    "inference.py",
    "model_weights.pth",
    "config.json",
    "reference_embeddings.npy",
    "reference_labels.npy",
    "requirements.txt",
    "README.md"
]


print("\n" + "=" * 75)
print("REMOTE FILE VERIFICATION")
print("=" * 75)


for file_name in required_files:

    file_path = (
        REMOTE_MAIZE_DIR /
        file_name
    )

    assert file_path.exists(), (
        f"Missing remote file: "
        f"{file_name}"
    )

    size_mb = (
        file_path.stat().st_size
        /
        1024
        /
        1024
    )

    print(
        f"✓ {file_name:<28} "
        f"{size_mb:>9.2f} MB"
    )


# ============================================================
# 8. REMOVE LOCALLY CACHED IMPORTS
#
# This ensures inference.py and model.py are loaded from the
# newly downloaded Hugging Face directory.
# ============================================================

for module_name in [
    "inference",
    "model"
]:

    if module_name in sys.modules:

        del sys.modules[
            module_name
        ]


# Remove previous occurrences from sys.path.
remote_directory_string = str(
    REMOTE_MAIZE_DIR
)

sys.path = [
    path
    for path in sys.path
    if path != remote_directory_string
]

# Put the downloaded directory first.
sys.path.insert(
    0,
    remote_directory_string
)


# ============================================================
# 9. IMPORT REMOTE INFERENCE PACKAGE
# ============================================================

importlib.invalidate_caches()

remote_inference_module = (
    importlib.import_module(
        "inference"
    )
)

MaizeDiseasePredictor = (
    remote_inference_module
    .MaizeDiseasePredictor
)


# ============================================================
# 10. LOAD REMOTE MODEL
# ============================================================

remote_predictor = (
    MaizeDiseasePredictor(
        model_dir=REMOTE_MAIZE_DIR
    )
)


print("\n" + "=" * 75)
print("REMOTE MODEL LOADED")
print("=" * 75)

print(
    "\n✓ model.py loaded from:",
    remote_inference_module.__file__
)

print(
    "✓ Model device:",
    remote_predictor.device
)

print(
    "✓ Reference count:",
    len(
        remote_predictor
        .reference_embeddings
    )
)

print(
    "✓ kNN K:",
    remote_predictor.knn_k
)

print(
    "✓ kNN threshold:",
    remote_predictor.knn_threshold
)


# ============================================================
# 11. UPLOAD EXACTLY ONE TEST IMAGE
# ============================================================

print(
    "\nUpload one genuine Maize image belonging to "
    "Healthy, MLN, MSV, NLB or NLS."
)

uploaded = files.upload()

if len(uploaded) != 1:

    raise ValueError(
        "Upload exactly one test image."
    )

uploaded_file_name = next(
    iter(
        uploaded
    )
)

TEST_IMAGE_PATH = Path(
    "/content"
) / uploaded_file_name

assert TEST_IMAGE_PATH.exists(), (
    f"Uploaded image not found: "
    f"{TEST_IMAGE_PATH}"
)


# ============================================================
# 12. RUN REMOTE INFERENCE
# ============================================================

result = remote_predictor.predict(
    TEST_IMAGE_PATH
)


# ============================================================
# 13. VERIFY RESPONSE CONTRACT
# ============================================================

assert result[
    "decision"
] in {
    "ACCEPT",
    "UNCERTAIN"
}

assert "predicted_class_code" in result
assert "predicted_class" in result
assert "confidence" in result
assert "reliability_score" in result
assert "reason_codes" in result


if result["decision"] == "ACCEPT":

    assert (
        result[
            "predicted_class_code"
        ]
        is not None
    )

    assert (
        result[
            "predicted_class"
        ]
        is not None
    )

    assert (
        "top_predictions"
        in result
    )

else:

    assert (
        result[
            "predicted_class_code"
        ]
        is None
    )

    assert (
        result[
            "predicted_class"
        ]
        is None
    )

    assert (
        "top_predictions"
        not in result
    )


# ============================================================
# 14. DISPLAY FINAL RESULT
# ============================================================

print("\n" + "=" * 75)
print("REMOTE HUGGING FACE INFERENCE RESULT")
print("=" * 75)

print(
    json.dumps(
        result,
        indent=2,
        ensure_ascii=False
    )
)


print("\n" + "=" * 75)
print("REMOTE SMOKE TEST PASSED")
print("=" * 75)

print(
    "\n✓ Repository authentication passed"
)

print(
    "✓ Remote files downloaded successfully"
)

print(
    "✓ Remote model loaded independently"
)

print(
    "✓ Reference embeddings loaded"
)

print(
    "✓ Prediction response contract verified"
)

if result["decision"] == "ACCEPT":

    print(
        "✓ Internal code:",
        result[
            "predicted_class_code"
        ]
    )

    print(
        "✓ Display name:",
        result[
            "predicted_class"
        ]
    )

else:

    print(
        "✓ Uncertain input was safely rejected"
    )

✓ HF_TOKEN found
✓ Token value remains hidden

Authenticated account: viraj1920
Repository: viraj1920/plant-disease-models



Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]


REMOTE FILE VERIFICATION
✓ __init__.py                       0.00 MB
✓ model.py                          0.00 MB
✓ inference.py                      0.02 MB
✓ model_weights.pth               142.73 MB
✓ config.json                       0.00 MB
✓ reference_embeddings.npy         11.25 MB
✓ reference_labels.npy              0.04 MB
✓ requirements.txt                  0.00 MB
✓ README.md                         0.00 MB


/content/maize_hf_remote_test/maize/v1/model.py:63: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(



REMOTE MODEL LOADED

✓ model.py loaded from: /content/maize_hf_remote_test/maize/v1/inference.py
✓ Model device: cuda
✓ Reference count: 5762
✓ kNN K: 1
✓ kNN threshold: 0.99242216

Upload one genuine Maize image belonging to Healthy, MLN, MSV, NLB or NLS.


Saving DSC00025.JPG to DSC00025.JPG

REMOTE HUGGING FACE INFERENCE RESULT
{
  "crop": "maize",
  "model_version": "1.0.0",
  "decision": "ACCEPT",
  "predicted_class_code": "NLS",
  "predicted_class": "Northern Leaf Spot",
  "confidence": 0.982702,
  "probability_margin": 0.977126,
  "reliability_score": 0.993298,
  "reliability_threshold": 0.99242216,
  "reason_codes": [],
  "top_predictions": [
    {
      "class_code": "NLS",
      "class_name": "Northern Leaf Spot",
      "probability": 0.982702
    },
    {
      "class_code": "MLN",
      "class_name": "Maize Lethal Necrosis",
      "probability": 0.005577
    },
    {
      "class_code": "Healthy",
      "class_name": "Healthy",
      "probability": 0.004468
    }
  ],
  "message": "Prediction accepted: Northern Leaf Spot"
}

REMOTE SMOKE TEST PASSED

✓ Repository authentication passed
✓ Remote files downloaded successfully
✓ Remote model loaded independently
✓ Reference embeddings loaded
✓ Prediction response contract verified


In [17]:
# ============================================================
# LOAD HUGGING FACE TOKEN FROM COLAB SECRETS
# ============================================================

from google.colab import userdata

try:

    HF_TOKEN = userdata.get(
        "HF_maize_token"
    )

except Exception as error:

    print(
        "HF_maize_token could not be accessed."
    )

    print(
        "Open Colab Secrets and confirm:"
    )

    print(
        "1. The secret name is exactly HF_maize_token"
    )

    print(
        "2. Notebook access is enabled"
    )

    raise error


if not HF_TOKEN:

    raise ValueError(
        "HF_maize_token exists but is empty."
    )


print("✓ HF_maize_token found")
print("✓ Token value remains hidden")

✓ HF_maize_token found
✓ Token value remains hidden
